# Model 1 — Logistic Regression
**COEN 330 — Applied Machine Learning**

**Validation strategy:** 5-fold Stratified K-Fold on the training set (80%).  
**Primary metric:** Recall on `is_risky=1` — catch as many risky applicants as possible.  
**Test set (20%):** touched only in the final cell for held-out evaluation.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sys.path.insert(0, os.path.abspath('../src'))

from utils import SEED, MODELS_DIR, RESULTS_DIR, PLOTS_DIR
from preprocessing import run_full_pipeline
from logistic_regression import train_baseline, tune, cross_validate_model, save_model
from evaluate import evaluate, plot_confusion_matrix, tune_threshold, save_metrics

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

DATA_PATH = Path(os.path.abspath('../data/raw/dataset.csv'))

## 1. Load & Preprocess Data

Calls the existing preprocessing pipeline — 80/20 stratified split, preprocessor fit on train only.

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(f'Dataset not found at {DATA_PATH}')

X_train, X_test, y_train, y_test, feature_names, preprocessor = run_full_pipeline(str(DATA_PATH))

print(f'Train : {X_train.shape}  |  Test : {X_test.shape}')
print(f'Train is_risky=1 : {y_train.mean():.1%}  |  Test is_risky=1 : {y_test.mean():.1%}')

## 2. Baseline Model

Default LogisticRegression (C=1, L2, threshold=0.5) — no tuning.  
This is our reference point. We run 5-fold CV to get a stable estimate of its performance.

In [ ]:
baseline = train_baseline(X_train, y_train)

baseline_cv = cross_validate_model(baseline, X_train, y_train, cv_folds=5)

print('Baseline — 5-Fold CV results on training set:')
for metric, vals in baseline_cv.items():
    print(f"  {metric:<14} mean={vals['mean']:.4f}  std={vals['std']:.4f}  folds={[f'{v:.3f}' for v in vals['folds']]}")

## 3. Hyperparameter Tuning

GridSearchCV over `C` ∈ {0.001, 0.01, 0.1, 1, 10, 100} × penalty ∈ {L1, L2}.  
5-fold Stratified K-Fold on the training set — the test set is never used for selection.

In [ ]:
tuned_model, cv_results = tune(X_train, y_train, cv_folds=5)

In [ ]:
# Top 10 configurations by mean CV recall
cols = ['param_C', 'param_l1_ratio', 'mean_test_score', 'std_test_score', 'rank_test_score']
cv_results[cols].sort_values('rank_test_score').head(10).reset_index(drop=True)

## 4. Cross-Validate the Tuned Model

Re-run CV with the best hyperparameters to report per-fold stability — goes into the report.

In [ ]:
tuned_cv = cross_validate_model(tuned_model, X_train, y_train, cv_folds=5)

print('Tuned LR — 5-Fold CV results on training set:')
for metric, vals in tuned_cv.items():
    print(f"  {metric:<14} mean={vals['mean']:.4f}  std={vals['std']:.4f}  folds={[f'{v:.3f}' for v in vals['folds']]}")

## 5. Threshold Tuning

The default 0.5 threshold is not optimal when recall is the priority.  
We sweep thresholds on the **training set CV predictions** to pick one that achieves ≥85% recall while maximising precision.  
This threshold is then applied consistently to the test set.

In [ ]:
best_threshold = tune_threshold(
    tuned_model, X_train, y_train,
    min_recall=0.85,
    save_path=str(PLOTS_DIR / 'lr_threshold_tuning.png')
)

## 6. Final Evaluation on the Held-Out Test Set

> The test set is used **once** here — not for tuning, not for threshold selection.

In [ ]:
# Default threshold
test_metrics_05 = evaluate(
    tuned_model, X_test, y_test,
    threshold=0.5,
    label='Tuned LR — Test — threshold=0.50'
)

In [ ]:
# Tuned threshold
test_metrics = evaluate(
    tuned_model, X_test, y_test,
    threshold=best_threshold,
    label=f'Tuned LR — Test — threshold={best_threshold:.2f}'
)

In [ ]:
plot_confusion_matrix(
    tuned_model, X_test, y_test,
    threshold=best_threshold,
    title='Logistic Regression — Test Set',
    save_path=str(PLOTS_DIR / 'lr_confusion_matrix.png')
)

In [ ]:
coef_df = pd.DataFrame({
    'feature':     feature_names,
    'coefficient': tuned_model.coef_[0]
}).sort_values('coefficient')

fig, ax = plt.subplots(figsize=(8, max(4, len(feature_names) * 0.35)))
colors = ['tomato' if c > 0 else 'steelblue' for c in coef_df['coefficient']]
ax.barh(coef_df['feature'], coef_df['coefficient'], color=colors, edgecolor='white')
ax.axvline(0, color='black', lw=0.8)
ax.set_title('Logistic Regression — Feature Coefficients\n(positive → pushes toward Risky)')
ax.set_xlabel('Coefficient')
plt.tight_layout()
plt.savefig(str(PLOTS_DIR / 'lr_coefficients.png'))
plt.show()

print(coef_df.to_string(index=False))

## 8. Save Model & Results

In [ ]:
save_model(tuned_model)

save_metrics(
    {'model': 'Logistic Regression', 'split': 'test', **test_metrics},
    path=str(RESULTS_DIR / 'metrics_table.csv')
)

## 9. Results Summary

In [ ]:
print(f"{'Metric':<22} {'CV (train)':<14} {'Test (0.5)':<14} {f'Test ({best_threshold:.2f})'}")
print('-' * 66)
for key in ['accuracy', 'recall_1', 'precision_1', 'f1_1']:
    cv_val   = tuned_cv.get(key, {}).get('mean', float('nan'))
    t05_val  = test_metrics_05.get(key, float('nan'))
    best_val = test_metrics.get(key, float('nan'))
    print(f'{key:<22} {cv_val:<14.4f} {t05_val:<14.4f} {best_val:.4f}')